In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql import functions as F

In [2]:
# Start the Spark Session
spark = SparkSession.builder \
    .appName("TaxiTrajectoryAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Schema definition
schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("taxi_id", LongType(), True),
    StructField("timestamp", LongType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True)
])

# Read csv from local
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("/workspace/data/gps_cleaned.csv")

# Print Schema 
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 22:33:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)



In [3]:
trip_counts = df.groupBy("trip_id").count()

# Filter-out outliers
clean_trips = trip_counts.filter("count < 10000").drop("count")

df_clean = df.join(clean_trips, "trip_id", "inner")

# Aggregate paths by row
df_paths = df_clean.groupBy("trip_id").agg(
    F.collect_list(F.struct("timestamp", "latitude", "longitude")).alias("trajectory")
)


In [4]:
# Simple helper functions
import math
import heapq

def distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two GPS points in meters"""
    R = 6371000  # Earth radius in meters
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

print("Helper functions loaded")

Helper functions loaded


In [5]:
# Build simple road network from GPS data

# Create segments from trajectories (stream with toLocalIterator + cap)
segments = []
for row in df_paths.limit(2000).toLocalIterator():  # reverted to 1000
    trajectory = row.trajectory
    for i in range(len(trajectory) - 1):
        p1 = trajectory[i]
        p2 = trajectory[i + 1]
        segments.append((p1.latitude, p1.longitude, p2.latitude, p2.longitude))

print(f"Created {len(segments)} segments from GPS data")

# Build graph: nodes are GPS coordinates, edges are segments
graph = {}
nodes = {}
node_id = 0

def get_node_id(lat, lon):
    global node_id
    # Round coordinates to create grid (reverted to 3 decimals)
    lat_round = round(lat, 3)
    lon_round = round(lon, 3)
    key = (lat_round, lon_round)
    
    if key not in nodes:
        nodes[key] = node_id
        graph[node_id] = {}
        node_id += 1
    return nodes[key]

# Add edges to graph
for lat1, lon1, lat2, lon2 in segments:
    node1 = get_node_id(lat1, lon1)
    node2 = get_node_id(lat2, lon2)
    
    if node1 != node2:
        dist = distance(lat1, lon1, lat2, lon2)
        graph[node1][node2] = dist
        graph[node2][node1] = dist  # bidirectional

print(f"Built graph with {len(graph)} nodes")

# Create reverse mapping: node_id -> (lat, lon)
id_to_coord = {v: k for k, v in nodes.items()}

26/01/19 22:35:12 WARN TaskMemoryManager: Failed to allocate a page (134217728 bytes), try again.
[Stage 4:======================================================>  (26 + 1) / 27]

Created 86434 segments from GPS data
Built graph with 7146 nodes


In [6]:
# Dijkstra algorithm implementation

def dijkstra(start_node, end_node):
    """Find shortest path between two nodes"""
    if start_node not in graph or end_node not in graph:
        return [], float('inf')
    
    # Priority queue: (distance, node, path)
    pq = [(0, start_node, [start_node])]
    visited = set()
    
    while pq:
        current_dist, current_node, path = heapq.heappop(pq)
        
        if current_node in visited:
            continue
        
        visited.add(current_node)
        
        if current_node == end_node:
            return path, current_dist
        
        # Check neighbors
        for neighbor, edge_dist in graph[current_node].items():
            if neighbor not in visited:
                new_dist = current_dist + edge_dist
                new_path = path + [neighbor]
                heapq.heappush(pq, (new_dist, neighbor, new_path))
    
    return [], float('inf')

def find_nearest_node(lat, lon):
    """Find nearest node to given coordinates"""
    min_dist = float('inf')
    nearest = None
    
    for coord, node_id in nodes.items():
        d = distance(lat, lon, coord[0], coord[1])
        if d < min_dist:
            min_dist = d
            nearest = node_id
    
    return nearest

print("Dijkstra algorithm ready")

Dijkstra algorithm ready


In [7]:
# Route calculation between two lat/lon points

def calculate_route(start_lat, start_lon, end_lat, end_lon):
    """Calculate shortest route between two GPS coordinates"""
    print(f"Route from ({start_lat:.4f}, {start_lon:.4f}) to ({end_lat:.4f}, {end_lon:.4f})")
    
    # Find nearest nodes
    start_node = find_nearest_node(start_lat, start_lon)
    end_node = find_nearest_node(end_lat, end_lon)
    
    if start_node is None or end_node is None:
        return {"success": False, "message": "No nearby nodes found"}
    
    # Calculate path using Dijkstra
    path_nodes, total_distance = dijkstra(start_node, end_node)
    
    if not path_nodes:
        return {"success": False, "message": "No route found"}
    
    # Convert to coordinates
    path_coords = []
    for node in path_nodes:
        lat, lon = id_to_coord[node]
        path_coords.append({"lat": lat, "lon": lon})
    
    return {
        "success": True,
        "path": path_coords,
        "distance_meters": total_distance,
        "distance_km": total_distance / 1000,
        "num_points": len(path_coords)
    }

print("Route calculation function ready")

Route calculation function ready


In [8]:
import random

# Ensure the graph exists (run the graph-building cell first)
if not id_to_coord:
    raise ValueError("Graph not built yet. Run the previous cells first.")

# Pick two random distinct node IDs from the graph
node_ids = list(id_to_coord.keys())
start_node = random.choice(node_ids)
end_node = random.choice(node_ids)
while end_node == start_node:
    end_node = random.choice(node_ids)

# Map nodes to lat/lon
start_lat, start_lon = id_to_coord[start_node]
end_lat, end_lon   = id_to_coord[end_node]

print("Random start/end (from graph nodes):")
print(f"Start: ({start_lat:.5f}, {start_lon:.5f})  -> node {start_node}")
print(f"End:   ({end_lat:.5f}, {end_lon:.5f})  -> node {end_node}")

# Compute route
res = calculate_route(start_lat, start_lon, end_lat, end_lon)
print("\nResult:", res["success"])
if res["success"]:
    print(f"Distance: {res['distance_meters']:.0f} m ({res['distance_km']:.2f} km)")
    print(f"Points: {res['num_points']}")
    print("\nFull path coordinates:")
    for i, p in enumerate(res["path"], 1):
        print(f"{i:3d}: {p['lat']:.6f}, {p['lon']:.6f}")
else:
    print("Message:", res["message"])

Random start/end (from graph nodes):
Start: (41.15600, -8.61900)  -> node 641
End:   (41.26100, -8.56500)  -> node 7111
Route from (41.1560, -8.6190) to (41.2610, -8.5650)

Result: True
Distance: 11769 m (11.77 km)
Points: 53

Full path coordinates:
  1: 41.156000, -8.619000
  2: 41.156000, -8.618000
  3: 41.157000, -8.618000
  4: 41.158000, -8.617000
  5: 41.160000, -8.617000
  6: 41.162000, -8.617000
  7: 41.163000, -8.616000
  8: 41.164000, -8.615000
  9: 41.166000, -8.615000
 10: 41.167000, -8.614000
 11: 41.167000, -8.613000
 12: 41.167000, -8.612000
 13: 41.167000, -8.611000
 14: 41.168000, -8.611000
 15: 41.168000, -8.610000
 16: 41.169000, -8.609000
 17: 41.170000, -8.607000
 18: 41.171000, -8.606000
 19: 41.173000, -8.606000
 20: 41.174000, -8.604000
 21: 41.174000, -8.603000
 22: 41.174000, -8.601000
 23: 41.175000, -8.600000
 24: 41.177000, -8.600000
 25: 41.179000, -8.599000
 26: 41.179000, -8.598000
 27: 41.180000, -8.598000
 28: 41.182000, -8.597000
 29: 41.183000, -8.597